##### Data Ingestion: Petroleum Consumption & Prices (EIA API)

**Latency requirement:** The source (EIA) updates data weekly for both datasets. For 
this task, a batch load (one–time historical pull) is sufficient

**Data volume:**
- Consumption (`petroleum/cons/wpsup`): 6,447 rows for 2009–2026, 2,149,203 bytes 
  (~2.05 MB) in raw JSON format.
- Prices (`petroleum/pri/gnd`), filtered to national level (`duoarea=NUS`) at the 
  source via API facets: a much smaller pull than the full regional dataset.

Two datasets are used: weekly U.S. petroleum product consumption (Product Supplied) 
by product type (gasoline, diesel, jet fuel, etc.), and weekly national retail 
gasoline/diesel prices – both retrieved via the official EIA Open Data API and later 
joined for combined consumption/price analysis.

##### Design decisions

- **`requests` over `aiohttp`:** only a few sequential API calls (pagination) are 
  needed per dataset, not many parallel requests to different sources – async would 
  add complexity without benefit.
- **`tenacity` over a manual retry loop:** industry–standard library for retry logic – 
  declarative `@retry` decorator, easier to read/maintain than a hand–rolled loop.
- **`logging` over `print`:** standard practice – supports log levels (INFO/WARNING/ERROR) 
  and integrates more easily with future monitoring/alerting than raw print statements.

In [0]:
dbutils.library.restartPython()

In [0]:
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("eia_ingestion")

In [0]:
import requests
import json
from tenacity import retry, stop_after_attempt, wait_exponential

API_KEY = dbutils.secrets.get(scope="eia_api", key="eia–api–key")
BASE_URL = "https://api.eia.gov/v2/petroleum/cons/wpsup/data/"

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=8))
def fetch_page(offset, length=5000):
    params ={
        "frequency": "weekly",
        "data[0]": "value",
        "start": "2009–01–01",
        "end": "2026–08–21",
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "offset": offset,
        "length": length,
        "api_key": API_KEY,
    }
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()
    return response


def fetch_all_data(page_size=5000):
    all_data = []
    offset = 0
    total = None

    while total is None or offset < total:
        response = fetch_page(offset=offset, length=page_size)
        body = response.json()["response"]
        
        if total is None:
            total = int(body["total"])
            logger.info(f"Total rows available: {total}")
        
        all_data.extend(body["data"])
        offset += page_size
        logger.info(f"Fetched {len(all_data)}/{total} rows so far")

    return all_data, total

In [0]:
all_data, total = fetch_all_data()

total_bytes = len(json.dumps(all_data).encode("utf–8"))
logger.info(f"Total rows fetched: {len(all_data)}")
logger.info(f"Total raw JSON size: {total_bytes} bytes ({total_bytes / 1024:.1f} KB)")

##### Step 2: Save & Import as JSON

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS dbr_dev_ua5816bd.roksolana_shendiu770.raw_files;

In [0]:
output_path = "/Volumes/dbr_dev_ua5816bd/roksolana_shendiu770/raw_files/petroleum_raw.json"

with open(output_path, "w") as f:
    json.dump(all_data, f)

logger.info(f"Saved {len(all_data)} rows to {output_path}")


In [0]:
imported_df = spark.read.json(output_path)
imported_df.printSchema()

In [0]:
imported_df.show(5)

##### Step 3: Load into a Delta table

**Why `overwrite` and not `MERGE`:** This is a full historical batch load (not incremental) – 
every run fetches the complete 2009–2026 range, so there's no partial/new–only data to merge. 
`overwrite` keeps the notebook idempotent and simple. `MERGE` will become necessary in the 
upcoming Incremental Ingestion lab, once we start pulling only new weekly data.

In [0]:
target_table = "dbr_dev_ua5816bd.roksolana_shendiu770.petroleum_raw"
imported_df.write.format("delta").mode("overwrite").saveAsTable(target_table)

logger.info(f"Data written to Delta table: {target_table}")

In [0]:
%sql
SELECT * FROM dbr_dev_ua5816bd.roksolana_shendiu770.petroleum_raw LIMIT 10;

##### Step 4: Second dataset – Petroleum Prices

In [0]:
PRICES_URL = "https://api.eia.gov/v2/petroleum/pri/gnd/data/"

@retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=2, min=2, max=8))
def fetch_prices_page(offset, length=5000):
    params = {
        "frequency": "weekly",
        "data[0]": "value",
        "facets[duoarea][]": "NUS",
        "start": "2009–01–01",
        "end": "2026–08–24",
        "sort[0][column]": "period",
        "sort[0][direction]": "desc",
        "offset": offset,
        "length": length,
        "api_key": API_KEY,
    }
    response = requests.get(PRICES_URL, params=params, timeout=30)
    response.raise_for_status()
    return response

def fetch_all_prices(page_size=5000):
    all_prices = []
    offset = 0
    total = None

    while total is None or offset < total:
        response = fetch_prices_page(offset=offset, length=page_size)
        body = response.json()["response"]

        if total is None:
            total = int(body["total"])
            logger.info(f"Total price rows available: {total}")

        all_prices.extend(body["data"])
        offset += page_size
        logger.info(f"Fetched {len(all_prices)}/{total} price rows so far")

    return all_prices, total

all_prices, prices_total = fetch_all_prices()
logger.info(f"Total price rows fetched: {len(all_prices)}")

In [0]:
prices_df = spark.createDataFrame(all_prices)

prices_target_table = "dbr_dev_ua5816bd.roksolana_shendiu770.petroleum_prices_raw"
prices_df.write.format("delta").mode("overwrite").saveAsTable(prices_target_table)

logger.info(f"Prices data written to Delta table: {prices_target_table}")

##### Step 5: Spark DataFrame operations (select, filter, groupBy, join)

In [0]:
consumption_df = spark.table('dbr_dev_ua5816bd.''roksolana_shendiu770.petroleum_raw')

prices_df = spark.table("dbr_dev_ua5816bd.roksolana_shendiu770.petroleum_prices_raw")


In [0]:
consumption_s = consumption_df.select("period", "product–name", "value", "units")

consumption_s.show(5)


In [0]:
prices_s = prices_df.select("period", "product–name", "value", "units")

prices_s.show(5)

In [0]:
gasoline_consumption = consumption_s.filter(
    consumption_s["product–name"] == "Finished Motor Gasoline"
)
gasoline_consumption.show(5)

In [0]:
from pyspark.sql.functions import avg

avg_by_product = consumption_df.groupBy("product–name").agg(
    avg("value").alias("avg_weekly_consumption")
)

avg_by_product.show()

In [0]:
from pyspark.sql.functions import to_date, date_trunc

consumption_df = spark.table("dbr_dev_ua5816bd.roksolana_shendiu770.petroleum_raw") \
    .withColumn("period_date", to_date("period", "yyyy–MM–dd"))

prices_df = spark.table("dbr_dev_ua5816bd.roksolana_shendiu770.petroleum_prices_raw") \
    .withColumn("period_date", to_date("period", "yyyy–MM–dd"))

In [0]:
consumption_df.select("period").distinct().orderBy("period", ascending=False).show(15)
prices_df.select("period").distinct().orderBy("period", ascending=False).show(15)

%md
**Date alignment note:** Consumption data (`petroleum_raw`) reports `period` as the 
Friday ending each report week, while price data (`petroleum_prices_raw`) reports 
`period` as the following Monday (price collection day, per EIA's documentation). 
Verified across the full available range – the offset is a consistent +3 days with 
no exceptions. We align consumption dates by +3 days to match the corresponding 
price collection date for the same report week.

In [0]:
consumption_df.select("product–name").distinct().show(truncate=False)
prices_df.select("product–name").distinct().show(truncate=False)

**Product mapping note:** Only two consumption categories have a direct national–level 
price counterpart in the `pri/gnd` dataset – Gasoline and Diesel (mapped via `product_mapping`). 
Prices are filtered to `duoarea = "NUS"` (U.S. national level) to avoid duplicating each 
week across ~19 regional price series. 

In [0]:
prices_df.select("duoarea", "area–name").distinct().orderBy("duoarea").show(20, truncate=False)

In [0]:
from pyspark.sql.functions import to_date

prices_df = prices_df.withColumn("period_aligned", to_date("period", "yyyy–MM–dd"))

prices_df.printSchema()

In [0]:
from pyspark.sql.functions import date_add
from pyspark.sql.functions import col

consumption_df = consumption_df.withColumn("period_aligned", date_add("period_date", 3))

consumption_df.printSchema()

In [0]:
product_mapping = {
    "Finished Motor Gasoline": "Total Gasoline",
    "Distillate Fuel Oil": "No 2 Diesel",
}

from pyspark.sql.functions import create_map, lit, col
from itertools import chain

mapping_expr = create_map([lit(x) for x in chain(*product_mapping.items())])

consumption_mapped = consumption_df.withColumn(
    "mapped_price_product", mapping_expr[col("product–name")]
).filter(col("mapped_price_product").isNotNull())

joined_df = consumption_mapped.join(
    prices_df,
    (consumption_mapped["period_aligned"] == prices_df["period_aligned"]) &
    (consumption_mapped["mapped_price_product"] == prices_df["product–name"]),
    how="inner"
).select(
    consumption_mapped["period_aligned"],
    consumption_mapped["product–name"].alias("consumption_product"),
    consumption_mapped["value"].alias("consumption_value"),
    prices_df["product–name"].alias("price_product"),
    prices_df["value"].alias("price_value")
)

joined_df.show(10)

In [0]:
from pyspark.sql.functions import col

prices_national = prices_df.filter(col("duoarea") == "NUS")

joined_df = consumption_mapped.join(
    prices_df,
    (consumption_mapped["period_aligned"] == prices_df["period_aligned"]) &
    (consumption_mapped["mapped_price_product"] == prices_df["product–name"]),
    how="inner"
).select(
    consumption_mapped["period_aligned"],
    consumption_mapped["product–name"].alias("consumption_product"),
    consumption_mapped["value"].alias("consumption_value"),
    prices_df["product–name"].alias("price_product"),
    prices_df["value"].alias("value")
)

joined_df.orderBy("period_aligned", ascending=False).show(10)

In [0]:
from pyspark.sql.functions import round

avg_by_product = consumption_df.groupBy("product–name").agg(
    round(avg("value"), 2).alias("avg_weekly_consumption")
)

avg_by_product.orderBy("avg_weekly_consumption", ascending=False).show(truncate=False)

##### Step 6 – Data Quality Checks

In [0]:
consumption_total = consumption_df.count()
consumption_mapped_total = consumption_mapped.count()
joined_total = joined_df.count()

print(f"consumption_df (all products): {consumption_total}")
print(f"consumption_mapped (Gasoline/Diesel only): {consumption_mapped_total}")
print(f"joined_df (after price match): {joined_total}")
print(f"Rows dropped (no price match, e.g. Propane/Jet Fuel/Residual): {consumption_total – consumption_mapped_total}")

In [0]:
invalid_values = joined_df.filter(
    (col("consumption_value").cast("double") <= 0) | 
    (col("price_value").cast("double") <= 0)
)
invalid_count = invalid_values.count()
logger.info(f"Rows with invalid (<=0) values: {invalid_count}")

if invalid_count > 0:
    invalid_values.show(10)

In [0]:
prices_national.filter(
    (col("period_aligned") == "2026–01–12") & (col("product–name") == "No 2 Diesel")
).show(truncate=False)

In [0]:
duplicates = joined_df.groupBy("period_aligned", "consumption_product").count() \
    .filter(col("count") > 1)
duplicate_count = duplicates.count()
print(f"Duplicate period+product combinations: {duplicate_count}")

if duplicate_count > 0:
    duplicates.show(10)

In [0]:
prices_df.filter(
    (col("period_aligned") == "2012–12–17") & (col("product–name") == "No 2 Diesel")
).show(truncate=False)

In [0]:
prices_df.filter(
    (col("period_aligned") == "2019–10–21") & (col("product–name") == "No 2 Diesel")
).show(truncate=False)

**Data quality finding:** 2 exact duplicate rows were found in the source EIA prices 
data (2012–12–17 and 2019–10–21, No 2 Diesel) – identical values across all fields, 
likely an API–side data artifact. Removed via `.dropDuplicates()` before the join.

In [0]:
prices_df = prices_df.dropDuplicates()

joined_df = consumption_mapped.join(
    prices_df,
    (consumption_mapped["period_aligned"] == prices_df["period_aligned"]) &
    (consumption_mapped["mapped_price_product"] == prices_df["product–name"]),
    how="inner"
).select(
    consumption_mapped["period_aligned"],
    consumption_mapped["product–name"].alias("consumption_product"),
    consumption_mapped["value"].alias("consumption_value"),
    prices_df["product–name"].alias("price_product"),
    prices_df["value"].alias("price_value")
)

duplicates_after = joined_df.groupBy("period_aligned", "consumption_product").count() \
    .filter(col("count") > 1)
logger.info(f"Duplicate combinations after cleaning: {duplicates_after.count()}")

##### Step 7: Build star schema

**dim_date** – date dimension with derived attributes and a surrogate key.

In [0]:
from pyspark.sql.functions import year, quarter, month, date_format, monotonically_increasing_id

dim_date = joined_df.select("period_aligned").distinct() \
    .withColumn("year", year("period_aligned")) \
    .withColumn("quarter", quarter("period_aligned")) \
    .withColumn("month", month("period_aligned")) \
    .withColumn("month_name", date_format("period_aligned", "MMM")) \
    .withColumn("day_of_week", date_format("period_aligned", "EEEE")) \
    .withColumn("date_sk", monotonically_increasing_id())

dim_date.write.format("delta").mode("overwrite").saveAsTable(
    "dbr_dev_ua5816bd.roksolana_shendiu770.dim_date"
)

dim_date.orderBy("period_aligned", ascending=False).show(5)

In [0]:
from pyspark.sql.functions import lit, monotonically_increasing_id

dim_product = joined_df.select(
    col("consumption_product").alias("product")
).distinct().withColumn(
    "product_category", lit("Transport Fuel")
).withColumn(
    "product_sk", monotonically_increasing_id()
)

dim_product.write.format("delta").mode("overwrite").saveAsTable(
    "dbr_dev_ua5816bd.roksolana_shendiu770.dim_product"
)

dim_product.show(truncate=False)

In [0]:
dim_date_lookup = spark.table("dbr_dev_ua5816bd.roksolana_shendiu770.dim_date") \
    .select("period_aligned", "date_sk")

dim_product_lookup = spark.table("dbr_dev_ua5816bd.roksolana_shendiu770.dim_product") \
    .select("product", "product_sk")

fact_petroleum_snapshot = joined_df \
    .withColumn("consumption_value", col("consumption_value").cast("double")) \
    .withColumn("price_value", col("price_value").cast("double")) \
    .join(dim_date_lookup, on="period_aligned", how="inner") \
    .join(
        dim_product_lookup,
        joined_df["consumption_product"] == dim_product_lookup["product"],
        how="inner"
    ) \
    .withColumn(
        "estimated_daily_spend",
        round(col("consumption_value") * 1000 * 42 * col("price_value"), 0)
    ) \
    .select(
        "date_sk",
        "product_sk",
        "consumption_value",
        "price_value",
        "estimated_daily_spend"
    )

fact_petroleum_snapshot.write.format("delta").mode("overwrite").saveAsTable(
    "dbr_dev_ua5816bd.roksolana_shendiu770.fact_petroleum_snapshot"
)

fact_petroleum_snapshot.show(10)

In [0]:
%sql
SELECT COUNT(*) FROM dbr_dev_ua5816bd.roksolana_shendiu770.fact_petroleum_snapshot;

In [0]:
print(f"joined_df (right before dim/fact build): {joined_df.count()}")

In [0]:
print(f"dim_date rows: {spark.table('dbr_dev_ua5816bd.roksolana_shendiu770.dim_date').count()}")
print(f"dim_product rows: {spark.table('dbr_dev_ua5816bd.roksolana_shendiu770.dim_product').count()}")

##### Step 8: Reporting view for dashboard

A denormalized view joins the fact table with both dimensions, giving the dashboard 
a single, readable source without repeating the join logic there.

In [0]:
%sql
CREATE OR REPLACE VIEW dbr_dev_ua5816bd.roksolana_shendiu770.petroleum_reporting_view AS
SELECT
    d.period_aligned,
    d.year,
    d.quarter,
    d.month,
    d.month_name,
    p.product,
    p.product_category,
    f.consumption_value,
    f.price_value,
    f.estimated_daily_spend
FROM dbr_dev_ua5816bd.roksolana_shendiu770.fact_petroleum_snapshot f
JOIN dbr_dev_ua5816bd.roksolana_shendiu770.dim_date d ON f.date_sk = d.date_sk
JOIN dbr_dev_ua5816bd.roksolana_shendiu770.dim_product p ON f.product_sk = p.product_sk

In [0]:
%sql
SELECT * FROM dbr_dev_ua5816bd.roksolana_shendiu770.petroleum_reporting_view LIMIT 10;

In [0]:
%sql
DESCRIBE dbr_dev_ua5816bd.roksolana_shendiu770.petroleum_reporting_view;